# W2 Lab — Prompting and Reasoning: Chain-of-Thought and Examples

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_lab_prompting.ipynb)

**Goal.** Make a model more accurate by making it write its reasoning before the
answer (chain-of-thought), teach it format and procedure by showing worked examples
(few-shot), sample and vote when one reasoning path is not enough (self-consistency),
and verify every prompt change by scoring it on a fixed evalset with code.

Why this week: the notes (Ch. 2) locate the model's only workspace in the text it
generates, so what the prompt makes the model write before the answer decides the
accuracy. **Chain-of-thought (CoT)** = prompting that makes the model generate its
solution process before the final answer (Wei et al., 2022). **Few-shot prompting** =
placing worked examples in the prompt for the model to imitate.

The path: setup (1) → answer-only versus worked solution, graded by code (2) → the
text as the model's only workspace: silent thinking and answer-first (3) → a
code-graded eval: baseline → format fix → your CoT prompt, target 11 of 12 (4) →
worked examples (5) → self-consistency (6) → the limit of written reasoning (7) →
four graded assignments (8) → completion and submission (9).

*Runtime:* Google Colab, top-to-bottom, ~90 minutes. Every model call is written out
as in W1: a message list, one `create` call, and the reply read from
`response.choices[0].message.content`. No helper functions hide the call.

**This lab is collected.** Fill in Section 1.3 and follow Section 9 to submit.

Cells marked ✍️ ask for your own writing — a fill-in or a written prediction.
Lines marked *Try:* name a change to make to the prompt before moving on: make it,
rerun the cell, and compare the two outputs. The changes are the practice; a cell
run once and left alone teaches less than the same cell run three times with three
prompts.

*Sources:* Sections 2 and 3 reproduce Wei et al. (2022), *Chain-of-Thought Prompting*
(Figure 1 and the reasoning-after-answer ablation) on course-authored problems, with
the silent-thinking control from Anthropic's *Prompt Engineering Interactive
Tutorial*, ch. 6; Sections 5 and 7 adapt that tutorial's ch. 7–8 (parent bot, email
dataset, graders, and the hippo question verbatim); Section 4, Anthropic's *Prompt
Evaluations*, lesson 3 (dataset, prompts, graders verbatim); Section 6 applies Wang
et al. (2022), *Self-Consistency*, to the Section 4 eval — no source lab. The only adaptation is the course API standard, `aisuite` + a
pasted key, with no assistant-prefill turns. The Section 8 assignments are
course-authored in the style of the tutorial exercises.

## 1. Setup

### 1.1 Installation

`aisuite` exposes multiple providers (OpenAI, Anthropic) behind one interface, so the
same code runs whichever provider your key belongs to.

*Do:* run the cell (~30 seconds, once per session).

In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

An **API key** = the secret string that identifies your account to the provider and
bills usage to it (issuing steps: the API Setup guide on the course site). Do not
share the notebook with the key inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell. Nothing prints.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Submission identity

This lab is collected. The completion check in Section 9 prints these values, and a
blank name fails it.

*Do:* fill in your name and student ID, run the cell.

In [ ]:
STUDENT_NAME = ""
STUDENT_ID = ""

print(f"submitting as: {STUDENT_NAME or '(name missing)'} ({STUDENT_ID or 'ID missing'})")

### 1.4 Client and a first call

The call is the same one that opened W1: a list holding one `user` message, one
`create` call, and the reply text at `response.choices[0].message.content`. Every
cell in this notebook repeats this shape; the prompt string is the only part that
changes.

*Do:* run the cell and confirm the output is exactly `ready`.

In [ ]:
import aisuite

client = aisuite.Client()

messages = [{"role": "user", "content": "Reply with exactly: ready"}]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
response.choices[0].message.content

Any error here is a setup problem, not a code problem — recheck the API Setup guide
before continuing.

## 2. Answer-Only versus Worked Solution

**Chain-of-thought (CoT)** = prompting that makes the model generate its solution
process before the final answer (Wei et al., 2022; notes §2.3). This section
reproduces the measurement behind the technique: the same model on the same
problems, asked in two ways, scored by code.

### 2.1 The apple problem

The problem below is Figure 1 of Wei et al. (2022). A 2022 model answered 27 when
asked for the answer alone and 9 when made to write the solution first (notes §2.1).
The first cell demands only the number; the second demands the worked solution and
the answer on the last line.

*Do:* run both cells and compare the two replies.


In [ ]:
## Answer only
APPLES = ("A cafeteria has 23 apples. They use 20 for lunch and buy 6 more. "
          "How many apples do they have?")

messages = [
    {"role": "user", "content": APPLES + " Reply with only the number."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)


In [ ]:
## Worked solution first
messages = [
    {"role": "user", "content": APPLES + " Write the solution step by step, "
                                "then give the answer on the last line as ANSWER: <number>."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)


On a current model both replies are correct. The problem needs one intermediate
value (23 − 20 = 3), and a single prediction of a current model carries that much.
The failure that Wei et al. measured has not disappeared; it has moved to problems
with more steps and more digits (notes §2.1), and the next cells find it there.

*Try:* change 23, 20, and 6 to 2317, 1940, and 685 and rerun both cells. Does the
answer-only reply survive four-digit numbers?

### 2.2 Larger numbers, graded by code

The template below keeps the shape of the apple problem and adds one
multiplication: `crates × items − shipped + returned`. Python computes the correct
answer for each parameter set, so the grader is exact and the score does not depend
on reading replies by hand. The answer-only prompt and the worked-solution prompt
run over the same six problems.

*Do:* run the problem cell, then the two scoring cells, and compare the scores.


In [ ]:
import re

WAREHOUSE = ("A warehouse holds {crates} crates with {items} items each. "
             "{shipped} items are shipped out and {returned} items are returned. "
             "How many items are in the warehouse now?")

PARAMS = [
    {"crates": 238, "items": 147, "shipped": 519,  "returned": 284},
    {"crates": 463, "items": 129, "shipped": 1207, "returned": 356},
    {"crates": 317, "items": 284, "shipped": 2046, "returned": 173},
    {"crates": 592, "items": 173, "shipped": 3318, "returned": 907},
    {"crates": 154, "items": 396, "shipped": 1480, "returned": 622},
    {"crates": 729, "items": 218, "shipped": 4155, "returned": 231},
]

problems = []
for p in PARAMS:
    answer = p["crates"] * p["items"] - p["shipped"] + p["returned"]
    problems.append({"question": WAREHOUSE.format(**p), "answer": answer})

for problem in problems:
    print(f"{problem['answer']:>7}  <-  {problem['question']}")


In [ ]:
## Answer only
outputs_answer_only = []
for problem in problems:
    messages = [
        {"role": "user", "content": problem["question"] + " Reply with only the number."},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_answer_only.append(response.choices[0].message.content)

score_answer_only = 0
for problem, output in zip(problems, outputs_answer_only):
    match = re.search(r"\d[\d,]*", output)
    value = int(match.group().replace(",", "")) if match else None
    correct = value == problem["answer"]
    score_answer_only += correct
    print(f"{'PASS' if correct else 'FAIL'}  expected={problem['answer']:>7}  output={output!r}")

print(f"\nanswer only: {score_answer_only}/{len(problems)}")


In [ ]:
## Worked solution first
WORKED = " Write the solution step by step, then give the answer on the last line as ANSWER: <number>."

outputs_worked = []
for problem in problems:
    messages = [
        {"role": "user", "content": problem["question"] + WORKED},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_worked.append(response.choices[0].message.content)

score_worked = 0
for problem, output in zip(problems, outputs_worked):
    match = re.search(r"ANSWER:\s*(\d[\d,]*)", output)
    value = int(match.group(1).replace(",", "")) if match else None
    correct = value == problem["answer"]
    score_worked += correct
    print(f"{'PASS' if correct else 'FAIL'}  expected={problem['answer']:>7}  extracted={value}")

print(f"\nworked solution: {score_worked}/{len(problems)}")


The two scores come from the same model, the same six problems, and the same
grader. The only difference is whether the intermediate values (the product, then
the difference) were written into the text before the answer was produced (notes
§2.2). Exact scores vary by model and by run; the direction does not.

*Try:* print one answer-only miss next to the correct value. The wrong value is
usually close in magnitude and built from digits that appear in the problem, which
is the shape of an answer emitted before the computation finished. Then shrink
`crates` and `items` to two digits and rerun both scoring cells; the gap narrows as
the digits shrink.


## 3. The Text as the Model's Only Workspace

Section 2 showed that a written solution raises accuracy. This section tests the
explanation. When the model produces a token it reads only the text so far, so an
intermediate value exists for the model only once it has been written (notes §2.2).
Two prompts keep the reasoning but change whether, or where, it is written, and
each runs over the six problems of Section 2 with the same grader.

### 3.1 Thinking in silence

The instruction below asks for careful silent reasoning and then only the number.
If reasoning could take place outside the text, this prompt would score like the
worked solution.

*Do:* run the cell and compare the score with both scores of Section 2.


In [ ]:
## Told to think silently
SILENT = (" Think about it carefully in silence, do not write your reasoning, "
          "and reply with only the number.")

score_silent = 0
for problem in problems:
    messages = [
        {"role": "user", "content": problem["question"] + SILENT},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    match = re.search(r"\d[\d,]*", output)
    value = int(match.group().replace(",", "")) if match else None
    correct = value == problem["answer"]
    score_silent += correct
    print(f"{'PASS' if correct else 'FAIL'}  expected={problem['answer']:>7}  output={output!r}")

print(f"\nsilent thinking: {score_silent}/{len(problems)}   (answer only: {score_answer_only}, worked: {score_worked})")


The score matches the answer-only run, not the worked-solution run. Reasoning that
is not written is not performed (source: tutorial ch. 6).

### 3.2 Answer first, solution after

The second prompt asks for the same worked solution as Section 2, but with the
answer on the first line and the solution after it. Every intermediate value is
still written; none of it is written before the answer.

*Do:* run the cell and compare with the worked-solution score.


In [ ]:
## Answer first, then the solution
ANSWER_FIRST = (" Give the answer on the first line as ANSWER: <number>, "
                "then explain the solution step by step.")

score_answer_first = 0
for problem in problems:
    messages = [
        {"role": "user", "content": problem["question"] + ANSWER_FIRST},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    match = re.search(r"ANSWER:\s*(\d[\d,]*)", output)
    value = int(match.group(1).replace(",", "")) if match else None
    correct = value == problem["answer"]
    score_answer_first += correct
    print(f"{'PASS' if correct else 'FAIL'}  expected={problem['answer']:>7}  extracted={value}")

print(f"\nanswer first: {score_answer_first}/{len(problems)}   (answer only: {score_answer_only}, worked: {score_worked})")


The score falls back to the answer-only level. The solution that follows is written
after the answer already exists in the text, and a token once emitted cannot be
revised, so a solution that reaches a different value cannot change the graded
line. Wei et al. (2022) report the same result in their ablation: a chain of
thought placed after the answer performs like no chain of thought at all.

Sections 2 and 3 together fix what a chain of thought is: intermediate values
written into the text before the answer, not an explanation attached after it.
Every chain-of-thought prompt from here on places the solution before the answer,
and every grader reads the answer from the end of the reply, from an `ANSWER:` last
line or from `<answer>` tags after a `<thinking>` block (Section 4.5).

*Try:* append to the answer-first instruction: `"If the solution reaches a different
number, end with CORRECTION: <number>."` Rerun and count the corrections. Then decide
which line a grader should read, and what that decision assumes about the model.

Section 2 graded six problems whose answers Python computed. The next section
builds the general instrument: a fixed test set with known answers, a grader, and a
prompt improved against the score.


## 4. A Code-Graded Evaluation

**Code-graded evaluation** = scoring a prompt by running it over a fixed test set
with known answers and grading the outputs with code (source: *Prompt Evaluations*,
lesson 3; dataset and prompts verbatim). The task: from a statement about an animal,
answer how many legs it has. Several statements are deliberately tricky (a fox that
lost a leg and regrew two).

### 4.1 The eval set

Each item is a statement plus a **golden answer** = the known correct output the
grader compares against.

*Do:* run the cell and read the twelve statements; mark the ones you expect the
model to miss.

In [ ]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]
print(len(eval_data), "items")

### 4.2 The first prompt, on one item

The prompt is a template string with one placeholder; `.format` fills the statement
in. **Delimiters** = markers (here the `<animal_statement>` tags) that bound the data
inside a prompt so that the instructions and the pasted statement cannot be confused. Before running the whole set, the cell sends a single item (the fox) and shows
the raw reply.

*Do:* run the cell and read the reply. Note its shape (a bare number or a sentence)
and its value (5 is correct).

In [ ]:
PROMPT_V1 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have? Please respond with a number"""

item = eval_data[2]   # the fox

messages = [
    {"role": "user", "content": PROMPT_V1.format(statement=item["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
output = response.choices[0].message.content
output

*Try:* change `item` to `eval_data[0]` (the human) and `eval_data[10]` (the octopus
that lost two legs and regrew three). Does the shape of the reply stay the same
across items?

### 4.3 The whole set, graded

The loop repeats the single call of 4.2 for every item and keeps the replies in a
list.

*Do:* run the first cell and read each reply next to its golden answer, then run the
grading cell.

In [ ]:
outputs_v1 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V1.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v1.append(response.choices[0].message.content)

for item, output in zip(eval_data, outputs_v1):
    print(f"golden={item['golden_answer']:>2}  output={output!r}")

The grader is one comparison: the reply, stripped of whitespace, must equal the
golden answer exactly. A correct number inside a sentence grades as wrong.

In [ ]:
score_v1 = 0
for item, output in zip(eval_data, outputs_v1):
    correct = output.strip() == item["golden_answer"]
    score_v1 += correct
    print(f"{'PASS' if correct else 'FAIL'}  golden={item['golden_answer']:>2}  output={output!r}")

print(f"\nscore: {score_v1}/12")

Two separate problems: sentences instead of bare numbers (a formatting failure — the
right value graded wrong), and wrong numbers on the tricky statements (a reasoning
failure). The next two prompts fix them one at a time.

### 4.4 Fixing the format

One changed line at the end of the prompt: *"Respond only with a numeric digit, like
2 or 6, and nothing else."*

*Do:* run the cell. Formatting failures should disappear; the tricky items should
still miss.

In [ ]:
PROMPT_V2 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have? Respond only with a numeric digit, like 2 or 6, and nothing else."""

outputs_v2 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V2.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v2.append(response.choices[0].message.content)

score_v2 = 0
for item, output in zip(eval_data, outputs_v2):
    correct = output.strip() == item["golden_answer"]
    score_v2 += correct
    print(f"{'PASS' if correct else 'FAIL'}  golden={item['golden_answer']:>2}  output={output!r}")

print(f"\nscore: {score_v2}/12")

*Try:* soften the last line to `"Respond with the number."` and rerun. Which items
come back as sentences again? The exact wording of a format instruction is part of
the prompt's specification, and the eval detects when it drifts.

### 4.5 Chain of thought, graded ✍️ (core)

The remaining misses are reasoning failures; Section 2's device is the candidate fix.
Two things change in the prompt: the model is told to reason step by step inside
`<thinking>` tags first, and to put the final answer — just the integer — inside
`<answer>` tags. The grader then needs one more step, extracting the integer from
the tags, because the reply now contains the reasoning as well.

*Do:* write `PROMPT_V3`. Keep the task statement and the `{statement}` placeholder;
replace the closing question with the two-part instruction (think in `<thinking>`
tags, then answer in `<answer>` tags with the integer alone). Then run the three
cells that follow.

Hints: name both tags explicitly; say what goes inside each; state that the
`<answer>` tags contain the number and nothing else — the extraction below takes
the tag content verbatim, so "5 legs" grades as wrong.

In [ ]:
### FILL IN (START) ###
PROMPT_V3 = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have?"""
### FILL IN (END) ###

One item first, raw. The reply should now be a `<thinking>` block followed by an
`<answer>` block.

In [ ]:
item = eval_data[2]   # the fox

messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=item["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
output = response.choices[0].message.content
print(output)

The grader cannot compare this reply as a whole with `"5"`. A regular expression
picks out what sits between the answer tags; the cell shows the match object and
the extracted string.

In [ ]:
import re

match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
print("match    :", match)
print("extracted:", match.group(1).strip() if match else None)

The extraction step is now part of the loop. Target: **at least 11 of 12**.

*Do:* run the cell; iterate on `PROMPT_V3` (rerun all three cells) until the target
is reached.

In [ ]:
outputs_v3 = []
for item in eval_data:
    messages = [
        {"role": "user", "content": PROMPT_V3.format(statement=item["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    outputs_v3.append(response.choices[0].message.content)

cot_score = 0
for item, output in zip(eval_data, outputs_v3):
    match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
    extracted = match.group(1).strip() if match else None
    correct = extracted == item["golden_answer"]
    cot_score += correct
    print(f"{'PASS' if correct else 'FAIL'}  golden={item['golden_answer']:>2}  extracted={extracted!s:>4}  {item['animal_statement'][:58]}")

print(f"\nscore: {cot_score}/12   (target: >= 11)")

The source run reached 12/12 with this prompt shape. The eval made a prompting claim,
"thinking step by step helps here", checkable, and the same sequence — baseline,
format fix, reasoning fix, each scored — is how prompts are improved in practice.

*Try:* two edits to `PROMPT_V3`, each followed by the three cells. First, remove the
words that restrict the answer tags to the number alone; watch the extraction pick up
"5 legs". Second, reverse the order: answer tags first, thinking tags after. The
answer is then produced before any reasoning has been written, and the tricky items
return to their 4.4 values (→ 3.2).

### 4.6 The price of thinking

Every response object carries a `usage` field with the token counts billed for that
call. The two cells send the fox item under the direct prompt and under the CoT
prompt and read the completion tokens off each response.

*Do:* run both cells and compare the completion token counts.

In [ ]:
## Direct answer
messages = [
    {"role": "user", "content": PROMPT_V2.format(statement=eval_data[2]["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)
response.usage.completion_tokens

In [ ]:
## Chain of thought
messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=eval_data[2]["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)
response.usage.completion_tokens

The written solution costs tokens; the accuracy of 4.5 was bought with them. This
exchange of output tokens for accuracy is test-time compute (notes §2.6), and
Section 6 spends more of it.

## 5. Learning from Examples

**Few-shot prompting** = placing example question–answer pairs in the prompt; the
model imitates their format, tone, and procedure. "Zero-shot", "one-shot", "n-shot"
count the examples. Often it is easier to show than to describe (source: tutorial
ch. 7).

### 5.1 The parent bot

A bot for children's questions. Asked cold, the model answers like an encyclopedia.

*Do:* run both cells and compare the tone.

In [ ]:
## Asked cold
messages = [
    {"role": "user", "content": "Will Santa bring me presents on Christmas?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## One worked example in the prompt
messages = [
    {"role": "user", "content": """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

One example fixed tone and length at once; no instruction described either.

*Try:* rewrite the example answer as a curt, factual parent (`"A: No. The tooth
fairy is a story."`) and rerun; the reply copies the new tone. Then add a second Q/A
pair in the warm voice and see whether two examples hold the tone more firmly than
one.

### 5.2 Email classification by examples ✍️ (tutorial exercise 7.1)

Classify customer emails into

- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

The grader checks the **last character** of the output against the correct letter.
The starter prompt does not even name the categories, so the loop scores 0 of 4.
Rewrite `EMAIL_PROMPT` so that the category list plus a few worked examples of
emails with correctly formatted answers make every output end with the right
letter. Target: **4 of 4**.

*Do:* rewrite `EMAIL_PROMPT`, run the cell, and iterate until all four emails read
`PASS`.

Hints: list the four categories; give two or three example emails (not the ones in
`EMAILS`) each answered with the same closing line, for instance
`The correct category is: B`; end the prompt with the same line left open
(`The correct category is:`) so the model completes it. The source's solution puts
the examples in the user prompt, as here.

In [ ]:
### FILL IN (START) ###
EMAIL_PROMPT = """Please classify this email as either green or blue: {email}"""
### FILL IN (END) ###

EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.",  # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?",  # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???",  # (C) Billing question
    "How did I get here I am not good with computer.  Halp.",  # (D) Other (please explain)
]
ANSWERS = [["B"], ["A", "D"], ["C"], ["D"]]

email_score = 0
for email, accepted in zip(EMAILS, ANSWERS):
    messages = [
        {"role": "user", "content": EMAIL_PROMPT.format(email=email)},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    correct = output.strip()[-1] in accepted
    email_score += correct
    print(f"{'PASS' if correct else 'FAIL'}  expected {'/'.join(accepted)}  got: ...{output.strip()[-40:]!r}")

print(f"\nscore: {email_score}/4")

*Try:* once the loop passes, delete the examples and keep only the category list and
the closing line. Does the format survive on instructions alone? Then add a fifth
email of your own to `EMAILS` (with its accepted letters in `ANSWERS`) and see
whether your examples generalize to it.

The source's chapter 6 solves this task with instructions alone; examples reach the
same format with less instruction-writing, and the two techniques compose. Worked
examples *of step-by-step solutions* are the few-shot CoT exemplars of Wei et al.
(2022) (notes §2.3, Method 2); Assignment 8.4 asks you to write them.

## 6. Self-Consistency

**Self-consistency** = sampling several reasoning paths for the same question at
nonzero temperature and taking the majority of the extracted answers: wrong paths
scatter, correct paths agree (procedure of Wang et al., 2022, applied to the
Section 4 eval; no source lab).

### 6.1 One sample at temperature 1.0

The call is 4.5's call on the fox with `temperature=1.0`; the cell shows the
extracted answer of one sample.

*Do:* run the cell three or four times and note the extracted answers.

In [ ]:
HARD = eval_data[2]   # the fox that lost a leg and regrew two

messages = [
    {"role": "user", "content": PROMPT_V3.format(statement=HARD["animal_statement"])},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0)
output = response.choices[0].message.content
match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
print(output)
print("\nextracted:", match.group(1).strip() if match else None)

At temperature 1.0 the reasoning path differs from run to run, and on a tricky item
the extracted answer sometimes differs too. One sample is one draw from the set of
possible paths.

### 6.2 Five samples

The loop repeats the call of 6.1 five times and keeps the extracted answers.

*Do:* run the cell and read the five values.

In [ ]:
N_SAMPLES = 5

samples = []
for i in range(N_SAMPLES):
    messages = [
        {"role": "user", "content": PROMPT_V3.format(statement=HARD["animal_statement"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=1.0)
    output = response.choices[0].message.content
    match = re.search(r"<answer>(.*?)</answer>", output, re.DOTALL)
    samples.append(match.group(1).strip() if match else None)

samples

### 6.3 The vote

`Counter` tallies the samples; the most common value is the answer.

*Do:* run the cell and compare the majority with the golden answer.

In [ ]:
from collections import Counter

votes = Counter(s for s in samples if s is not None)
majority = votes.most_common(1)[0][0] if votes else None

print("votes   :", dict(votes))
print("majority:", majority, "  golden:", HARD["golden_answer"])

A single sampled run stands or falls with one path; the vote marginalizes over paths.
The price is five calls instead of one — the accuracy-for-compute exchange named
test-time compute (notes §2.6), which Ch. 11's theory follows into the model's own
training.

*Try:* set `temperature=0.0` in 6.2 and rerun 6.2 and 6.3; the five samples collapse
to five copies of one path, and the vote has nothing to marginalize. Restore 1.0
and set `N_SAMPLES = 9`; does a larger vote change the majority? Finally point
`HARD` at `eval_data[0]` (the human) and rerun: on an item every path gets right, the
vote is five calls spent for nothing — which questions deserve the extra samples is
the decision Ch. 12 takes up.

## 7. The Limit of Written Reasoning

Chain-of-thought repairs computation, not knowledge. When the model does not know a
fact, writing more does not produce it; the model generates a plausible premise and
reasons coherently on top of it (notes §2.4). The question below has no reliable
answer in any source (source: tutorial ch. 8).

*Do:* run the three cells and compare what each prompt produces.

In [ ]:
## Asked cold
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## With chain of thought
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time? "
                                "Think step by step in <thinking> tags first, then give your answer."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

In [ ]:
## With an out
messages = [
    {"role": "user", "content": "Who is the heaviest hippo of all time? "
                                "Only answer if you know the answer with certainty."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

The chain-of-thought version reasons in good form toward a name it cannot know; the
form of the reasoning does not check the premise. **Giving an out** = an instruction
that makes declining an acceptable answer, so that the model's default of being
helpful does not turn into invention. What checks a premise against the world is a
tool, and that is W3.

*Try:* replace the out with `"If you are not sure, answer exactly: I don't know."`
and rerun. Then ask, with the same out, a question the model does know ("What is the
largest land animal alive today?") and confirm that the out does not suppress a known
answer.

## 8. Assignments ✍️ (graded)

Four assignments, each with a fixed check cell. The code around the fill-in stays
as it is; only the prompt string, or the sampling you add, changes until the check
prints `PASS`. Grading reads the saved output of each check cell and of the Section 9
completion cell, so the notebook must be run top to bottom before submission.

| assignment | technique | notes |
|---|---|---|
| 8.1 | chain-of-thought by instruction | §2.3, Method 1 |
| 8.2 | format pinned by a worked example | §2.3, Method 2 |
| 8.3 | self-consistency, written by you | §2.5 |
| 8.4 | chain-of-thought by worked examples | §2.3, Method 2 |

### 8.1 An instruction that elicits the solution ✍️

`COT_INSTRUCTION` is appended after each word problem. The starter is empty, so the
model answers in whatever shape it likes and the extraction finds no `ANSWER:` line.
Both problems must pass.

*Do:* write the instruction, run the cell, iterate until both lines read `PASS`.

Hints: demand the solution step by step, and state the exact final line —
`ANSWER: <number>`. The check extracts that line and compares the number.

In [ ]:
WORD_PROBLEMS = [
    {"question": "A library has 4 shelves with 38 books each. During the day 47 books "
                 "are checked out and 26 are returned. How many books are on the shelves now?",
     "answer": "131"},
    {"question": "A bakery bakes 12 trays of 24 muffins. 6 trays are sold whole and 37 more "
                 "muffins are sold singly. How many muffins remain?",
     "answer": "107"},
]

### FILL IN (START) ###
COT_INSTRUCTION = ""
### FILL IN (END) ###

a1_passed = 0
for problem in WORD_PROBLEMS:
    messages = [
        {"role": "user", "content": problem["question"] + "\n" + COT_INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    match = re.search(r"ANSWER:\s*(\d+)", output)
    extracted = match.group(1) if match else None
    correct = extracted == problem["answer"]
    a1_passed += correct
    print(f"{'PASS' if correct else 'FAIL'}  extracted: {extracted}   expected: {problem['answer']}")
    print("      output ended:", repr(output.strip()[-60:]))

assignment_1 = a1_passed == len(WORD_PROBLEMS)
print(f"\nassignment 8.1: {a1_passed}/{len(WORD_PROBLEMS)}")

### 8.2 An exemplar that pins the format ✍️

The check accepts exactly one output shape per note: `DATE: 2026-03-10` — nothing
before it, nothing after it. Instructions tend to leave something loose (a trailing
period, a sentence around the date); a worked exemplar states the shape by showing
it. Both notes must pass.

*Do:* add one or two worked note → answer exemplars inside `DATE_PROMPT`, above the
test note, then run; iterate until both lines read `PASS`.

Hints: format each exemplar exactly as the output should look — for instance a note
about June 9th, 2026 answered with `DATE: 2026-06-09` on its own line — and keep the
test note last, laid out the same way as the exemplars. The exemplar notes must not
be the test notes.

In [ ]:
TEST_NOTES = [
    {"note": "Team offsite moved from March 3rd to March 10th, 2026.",
     "answer": "DATE: 2026-03-10"},
    {"note": "Deadline for the grant report extended from April 1st to April 22nd, 2026.",
     "answer": "DATE: 2026-04-22"},
]

### FILL IN (START) ###
DATE_PROMPT = """Extract the final date from the note.

Note: {note}"""
### FILL IN (END) ###

a2_passed = 0
for test in TEST_NOTES:
    messages = [
        {"role": "user", "content": DATE_PROMPT.format(note=test["note"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    correct = output.strip() == test["answer"]
    a2_passed += correct
    print(f"{'PASS' if correct else 'FAIL'}  got {output.strip()!r}   expected {test['answer']!r}")

assignment_2 = a2_passed == len(TEST_NOTES)
print(f"\nassignment 8.2: {a2_passed}/{len(TEST_NOTES)}")

### 8.3 The vote, written by you ✍️

Section 6 demonstrated self-consistency with given code. Here the statement is new —
a spider that lost three legs, golden answer 5 — and the sampling loop is yours.

*Do:* fill in the loop so that `samples` holds five extracted answers: five runs of
your `PROMPT_V3` on `TRICKY` at `temperature=1.0`, each passed through the same
regular expression as Section 6. The vote below is already written. Iterate until
`PASS`.

Hints: the pattern is the cell of 6.2 with `HARD` replaced by `TRICKY`. Keep
`temperature=1.0`; at 0.0 the five samples are five copies of a single path. The
check also requires at least five samples.

In [ ]:
TRICKY = {"animal_statement": "The animal is a spider that lost three legs.",
          "golden_answer": "5"}

### FILL IN (START) ###
samples = []
### FILL IN (END) ###

votes = Counter(s for s in samples if s is not None)
majority = votes.most_common(1)[0][0] if votes else None
assignment_3 = len(samples) >= 5 and majority == TRICKY["golden_answer"]

print("samples :", samples)
print(f"{'PASS' if assignment_3 else 'FAIL'}  majority: {majority}   golden: {TRICKY['golden_answer']}   samples: {len(samples)}")

### 8.4 Worked examples that elicit the solution ✍️

The evalset below holds six arithmetic expressions in Roman numerals. The check is
strict on two counts: the extracted number must be right, and the **last line** of
the reply must be exactly `ANSWER: <number>`. The starter prompt gives the
instruction alone; the shape of the solution and of the final line is left to the
model, and the last-line check tends to fail.

Write `ROMAN_PROMPT` in the exemplar form of chain-of-thought (notes §2.3, Method 2):
at least two worked examples, each converting the numerals, computing, and closing
with `ANSWER: <number>` on its own line, followed by the test expression laid out
the same way. Target: **at least 5 of 6**, and no evalset expression may appear in
the prompt (the check reports a leak).

*Do:* write the exemplars, run the cell, iterate until the line reads `PASS`.

Hints: choose exemplar expressions that are not in `ROMAN_EVALSET` (for instance
`XII + VII`); write each solution the way you want the model's solution to look —
numeral, value, computation, `ANSWER:` line; keep the placeholder `{expression}`
last.

In [ ]:
ROMAN_EVALSET = [
    {"expression": "XIV + IX", "answer": "23"},
    {"expression": "XLII - XXIX", "answer": "13"},
    {"expression": "CD + XC", "answer": "490"},
    {"expression": "MMXXVI - MCMXCIV", "answer": "32"},
    {"expression": "XCIX + I", "answer": "100"},
    {"expression": "LXXX / XVI", "answer": "5"},
]

### FILL IN (START) ###
ROMAN_PROMPT = """Evaluate the expression written in Roman numerals. Give the result as ANSWER: <number>.

{expression}"""
### FILL IN (END) ###

leaked = [e["expression"] for e in ROMAN_EVALSET if e["expression"] in ROMAN_PROMPT]
n_exemplars = len(re.findall(r"^ANSWER:\s*\d+\s*$", ROMAN_PROMPT, re.M))   # answer lines of the worked examples

roman_score = 0
for item in ROMAN_EVALSET:
    messages = [
        {"role": "user", "content": ROMAN_PROMPT.format(expression=item["expression"])},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    last_line = output.strip().splitlines()[-1].strip()
    correct = last_line == f"ANSWER: {item['answer']}"
    roman_score += correct
    print(f"{'PASS' if correct else 'FAIL'}  {item['expression']:18} last line: {last_line!r}")

assignment_4 = roman_score >= 5 and n_exemplars >= 2 and not leaked
print(f"\nassignment 8.4: {roman_score}/6   exemplars: {n_exemplars}   leaked: {leaked or 'none'}")
print("PASS" if assignment_4 else "FAIL")

## 9. Completion and Submission

This lab is collected. Completion criteria: the identity line is filled and all four
assignments pass. Sections 4.5 and 5.2 are in-class practice and are not graded.
Grading checks these structural facts, never prose quality.

*Do:* run the notebook top to bottom once more so every output is saved, confirm
every row below reads `PASS`, then download the notebook (**File → Download →
Download .ipynb**) and submit it the way announced in class.

In [ ]:
completion = {
    "name and student ID filled in (1.3)":            bool(STUDENT_NAME.strip()) and bool(STUDENT_ID.strip()),
    "8.1 instruction elicits ANSWER on both problems": assignment_1,
    "8.2 exemplar pins the format on both notes":      assignment_2,
    "8.3 the vote finds the answer":                   assignment_3,
    "8.4 worked examples reach >= 5/6":                assignment_4,
}
print(f"submitted by: {STUDENT_NAME or '?'} ({STUDENT_ID or '?'})\n")
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE — download and submit" if all(completion.values()) else "\nNOT COMPLETE YET")

---

Reference fill-ins: `labs/checkpoints/week02/solution.py` (lab and homework
together), published after the homework deadline.

W3 turns plain Python functions into tools the model can call (DeepLearning.AI,
*Agentic AI*, Module 3): function calling, the request–execute–reinject trace, and a
measured routing score.